# SetBased Model on Cranfield Collection

This notebook demonstrates the SetBased information retrieval model on the Cranfield (CF) collection.

**Model:** Pure set-based retrieval with Apriori termset mining and IDF weighting, no graph-based node weighting.

**Collection:** CF has 1,239 documents and 100 queries with relevance judgments.

## Setup

In [22]:
from pathlib import Path
from graph_ir.data.collection import Collection
from graph_ir.models.setbased import SetBasedModel
import numpy as np

# Locate the repo root regardless of where Jupyter's working directory is
# (it is the notebook's folder by default, not the repo root).
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root (no pyproject.toml found)")

REPO_ROOT = find_repo_root(Path.cwd())
cf_path = REPO_ROOT / "collections" / "cf"
print(f"Loading CF from {cf_path}...")

Loading CF from /home/nikitas/PycharmProjects/Graph_ir/collections/cf...


## Load Collection

In [23]:
# Load collection (load first 10 queries for a quick demo)
collection = Collection(cf_path).create()
collection.load_queries().load_qrels()

print(f"\nCollection stats:")
print(f"  Documents: {collection.num_docs}")
print(f"  Queries: {len(collection.queries)}")
print(f"  Vocabulary: {len(collection.inverted_index)} terms")
print(f"  Relevance judgments: {len(collection.qrels)} queries with qrels")

# Sample query
sample_qid = list(collection.queries.keys())[0]
print(sample_qid)
sample_query = collection.queries[sample_qid]
print(sample_query)
print(f"\nSample query {sample_qid}: {sample_query[:60]}...")


Collection stats:
  Documents: 1239
  Queries: 100
  Vocabulary: 18829 terms
  Relevance judgments: 100 queries with qrels
1
What are the effects of calcium on the physical properties of mucus from CF patients?

Sample query 1: What are the effects of calcium on the physical properties o...


## Initialize SetBased Model

In [24]:
model = SetBasedModel(collection)
print(f"Model: {model.get_model()}")
print(f"Vectorizer: IDF * termset frequency")
print(f"Weighting: Uniform (no graph-based node weights)")

Model: SetBasedModel
Vectorizer: IDF * termset frequency
Weighting: Uniform (no graph-based node weights)


## Fit Model (Mine Termsets)

This step runs Apriori on each query to find frequent termsets across documents.

In [25]:
import time

# Fit every loaded query with minimum support of 1 document.
print("Fitting model (Apriori termset mining)...")
start = time.time()

model.fit(min_freq=1, stopwords=False)

elapsed = time.time() - start
print()
print(f"Fit complete in {elapsed:.2f}s")
print(f"Queries fitted: {len(model._query_ids)}")
print(f"Total termsets mined: {sum(len(ts) for ts in model._docVectors)}")

Fitting model (Apriori termset mining)...

Fit complete in 4.01s
Queries fitted: 100
Total termsets mined: 188185


## Evaluate Rankings

In [26]:
print("Evaluating...")
precisions, recalls = model.evaluate()

mean_prec = np.mean(precisions)
mean_rec = np.mean(recalls)

print(f"  Mean Average Precision: {mean_prec:.3f}")
print(f"  Mean Recall: {mean_rec:.3f}")


Evaluating...
  Mean Average Precision: 0.155
  Mean Recall: 0.524


## Inspect a Ranking

In [27]:
# Show top 5 documents for first query
if model.ranking:
    first_ranking = model.ranking[0]
    first_qid = model._query_ids[0]
    
    print(f"Top 5 documents for query {first_qid}:")
    print(f"Query: {collection.queries[first_qid][:70]}...\n")
    
    relevant = collection.relevant[first_qid]
    for rank, doc_id in enumerate(first_ranking[:5], start=1):
        is_rel = "✓ RELEVANT" if doc_id in relevant else "  -"
        print(f"  {rank}. Doc {doc_id:4s} {is_rel}")

Top 5 documents for query 1:
Query: What are the effects of calcium on the physical properties of mucus fr...

  1. Doc 392  ✓ RELEVANT
  2. Doc 437    -
  3. Doc 767    -
  4. Doc 441  ✓ RELEVANT
  5. Doc 568    -


## Summary

SetBased model successfully:
- Loads the CF collection (1,239 docs, 100 queries)
- Mines frequent termsets using Apriori per query
- Ranks documents by cosine similarity with IDF weighting
- Evaluates precision and recall at a given cutoff (k=10)

This is the foundation for graph-based variants (GSB, windowed GSB, spectral models) that add graph-derived node weighting.